# SWATCH 방호성능 회귀 분석

| 순서 | 방법 | 목적 |
|---|---|---|
| 1 | **변수 변환 + OLS** | 변환 조합별 계수 유의성(p-값) 비교 |
| 2 | **다항식 Ridge** | 비선형 항 추가, LOOCV로 예측력 평가 |
| 3 | **가우시안 프로세스 회귀 (GPR)** | 비모수 회귀 적합, LML 유의성 검정 |

> **주의**: n=9, 파라미터 최대 5개 → 자유도(df_residual)가 3~6 수준으로 매우 작음.  
> OLS p-값이 유의하기 어렵고, GPR의 LML 검정이 보완적 역할을 함.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import statsmodels.formula.api as smf
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

---
## 0. 데이터 로드 및 변환 변수 생성

In [ ]:
raw = pd.read_excel('./datasets/swatch.xlsx')

# clean 이름으로 변환 변수 생성 (statsmodels formula에서 Q() 불필요)
df = pd.DataFrame()
df['Sample'] = raw.iloc[:, 0]
df['x1']    = raw.iloc[:, 1]          # 평량(g/m2)
df['x2']    = raw.iloc[:, 2]          # BET(m2/g)
df['y_GD']  = raw.iloc[:, 3]          # SWATCH(GD) 24h(Con)
df['y_HD']  = raw.iloc[:, 4]          # SWATCH(HD) 24h(Con)
df['lx1']   = np.log(df['x1'])        # log(평량)
df['lx2']   = np.log(df['x2'])        # log(BET)
df['ly_GD'] = np.log(df['y_GD'])      # log(GD)
df['ly_HD'] = np.log(df['y_HD'])      # log(HD)

X_raw = df[['x1', 'x2']].values
feat_names = ['평량(g/m²)', 'BET(m²/g)']

df

In [ ]:
# 산점도: 각 특성 × 각 변환 조합
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col_i, (x_col, x_label) in enumerate([('x1','평량'), ('lx1','log(평량)'), ('x2','BET'), ('lx2','log(BET)')]):
    for row_i, (y_col, y_label) in enumerate([('y_GD','GD'), ('ly_GD','log(GD)')]):
        ax = axes[row_i, col_i]
        ax.scatter(df[x_col], df[y_col], color='steelblue', edgecolors='k', s=70)
        ax.set_xlabel(x_label, fontsize=9)
        ax.set_ylabel(y_label, fontsize=9)

fig.suptitle('EDA — GD 방호성능 (원값 vs 로그)', fontsize=12)
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(2, 4, figsize=(16, 7))
for col_i, (x_col, x_label) in enumerate([('x1','평량'), ('lx1','log(평량)'), ('x2','BET'), ('lx2','log(BET)')]):
    for row_i, (y_col, y_label) in enumerate([('y_HD','HD'), ('ly_HD','log(HD)')]):
        ax = axes2[row_i, col_i]
        ax.scatter(df[x_col], df[y_col], color='tomato', edgecolors='k', s=70)
        ax.set_xlabel(x_label, fontsize=9)
        ax.set_ylabel(y_label, fontsize=9)

fig2.suptitle('EDA — HD 방호성능 (원값 vs 로그)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 1. 변수 변환 + OLS 선형회귀

시도할 변환 조합 6가지:

| # | Y 변환 | X 변환 | 파라미터 수 |
|---|---|---|---|
| ① | 없음 | 없음 | 2 |
| ② | 없음 | log | 2 |
| ③ | log | 없음 | 2 |
| ④ | log | log (**log-log, power-law**) | 2 |
| ⑤ | log | log + 교호작용 | 3 |
| ⑥ | log | log + 2차항 | 4 |

> AIC 기준으로 최적 모형 선정 → 전체 summary 출력

In [ ]:
def ols_compare(df, formulas, y_raw_col):
    """
    formulas: list of (label, formula_str, is_log_y)
    반환: 비교 테이블, {label: (result, is_log_y)}
    """
    rows, results = [], {}
    for label, formula, is_log_y in formulas:
        res = smf.ols(formula, data=df).fit()
        y_fit = np.exp(res.fittedvalues) if is_log_y else res.fittedvalues
        y_true = df[y_raw_col]
        pvals = res.pvalues.drop('Intercept', errors='ignore')
        rows.append({
            '모형':        label,
            'R²':         round(res.rsquared, 4),
            'adj-R²':     round(res.rsquared_adj, 4),
            'AIC':        round(res.aic, 2),
            'F p-값':     f"{res.f_pvalue:.4f}",
            'min coef p': f"{pvals.min():.4f}",
            'df_resid':   int(res.df_resid),
            'RMSE(원값)': round(np.sqrt(mean_squared_error(y_true, y_fit)), 2)
        })
        results[label] = (res, is_log_y)
    return pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True), results


def plot_ols_fit(res, is_log_y, y_true, title):
    y_fit = np.exp(res.fittedvalues) if is_log_y else res.fittedvalues
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_true, y_fit, color='steelblue', edgecolors='k', s=70)
    lo, hi = min(y_true.min(), y_fit.min()), max(y_true.max(), y_fit.max())
    ax.plot([lo, hi], [lo, hi], 'r--')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Fitted')
    r2 = r2_score(y_true, y_fit)
    ax.set_title(f"{title}\nR²={r2:.4f}")
    plt.tight_layout()
    plt.show()

### 1-1. GD 방호성능

In [ ]:
formulas_GD = [
    ('① raw ~ x1+x2',              'y_GD  ~ x1 + x2',                             False),
    ('② raw ~ log(x)',              'y_GD  ~ lx1 + lx2',                           False),
    ('③ log(y) ~ x',               'ly_GD ~ x1 + x2',                              True),
    ('④ log(y) ~ log(x)',           'ly_GD ~ lx1 + lx2',                            True),
    ('⑤ log(y) ~ log(x)+교호작용',  'ly_GD ~ lx1 + lx2 + lx1:lx2',                 True),
    ('⑥ log(y) ~ log(x)+2차항',    'ly_GD ~ lx1 + lx2 + I(lx1**2) + I(lx2**2)',   True),
]

table_GD, results_GD = ols_compare(df, formulas_GD, 'y_GD')
print("GD 방호성능 — 변환 조합 비교 (AIC 오름차순)")
display(table_GD)

In [ ]:
# AIC 최소 모형 상세 결과
best_label_GD = table_GD.iloc[0]['모형']
best_res_GD, best_log_GD = results_GD[best_label_GD]

print(f"=== GD 최적 모형: {best_label_GD} ===")
print(best_res_GD.summary())

plot_ols_fit(best_res_GD, best_log_GD, df['y_GD'], f"GD 최적 OLS ({best_label_GD})")

### 1-2. HD 방호성능

In [ ]:
formulas_HD = [
    ('① raw ~ x1+x2',              'y_HD  ~ x1 + x2',                             False),
    ('② raw ~ log(x)',              'y_HD  ~ lx1 + lx2',                           False),
    ('③ log(y) ~ x',               'ly_HD ~ x1 + x2',                              True),
    ('④ log(y) ~ log(x)',           'ly_HD ~ lx1 + lx2',                            True),
    ('⑤ log(y) ~ log(x)+교호작용',  'ly_HD ~ lx1 + lx2 + lx1:lx2',                 True),
    ('⑥ log(y) ~ log(x)+2차항',    'ly_HD ~ lx1 + lx2 + I(lx1**2) + I(lx2**2)',   True),
]

table_HD, results_HD = ols_compare(df, formulas_HD, 'y_HD')
print("HD 방호성능 — 변환 조합 비교 (AIC 오름차순)")
display(table_HD)

In [ ]:
best_label_HD = table_HD.iloc[0]['모형']
best_res_HD, best_log_HD = results_HD[best_label_HD]

print(f"=== HD 최적 모형: {best_label_HD} ===")
print(best_res_HD.summary())

plot_ols_fit(best_res_HD, best_log_HD, df['y_HD'], f"HD 최적 OLS ({best_label_HD})")

---
## 2. 다항식 Ridge 회귀

- 2차 다항식 항(x², 교호작용 포함) + StandardScaler + Ridge
- LOOCV(n=9)로 최적 α 선택
- **주의**: Ridge는 정규화 회귀로 p-값이 없음. 예측력 평가에 사용.

In [ ]:
def fit_ridge_poly(X_raw, y_raw, log_y=False, label=''):
    """
    Poly(deg=2) + Ridge, LOOCV alpha 선택.
    log_y=True 이면 log(y)를 목표로 학습하고 원값으로 역변환.
    """
    y_target = np.log(y_raw) if log_y else y_raw

    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge',  Ridge())
    ])
    loo = LeaveOneOut()
    grid = GridSearchCV(
        pipe,
        {'ridge__alpha': np.logspace(-2, 5, 300)},
        cv=loo, scoring='neg_mean_squared_error'
    )
    grid.fit(X_raw, y_target)
    best = grid.best_estimator_
    alpha = grid.best_params_['ridge__alpha']

    # In-sample 적합값
    y_fit_sc = best.predict(X_raw)
    y_fit = np.exp(y_fit_sc) if log_y else y_fit_sc

    # LOOCV 예측값 (원값 스케일)
    loo_sc = cross_val_predict(best, X_raw, y_target, cv=loo)
    loo_pred = np.exp(loo_sc) if log_y else loo_sc

    insample_r2  = r2_score(y_raw, y_fit)
    loo_rmse     = np.sqrt(mean_squared_error(y_raw, loo_pred))
    loo_r2       = r2_score(y_raw, loo_pred)

    feat_names = best.named_steps['poly'].get_feature_names_out(['평량', 'BET'])
    coefs      = best.named_steps['ridge'].coef_
    intercept  = best.named_steps['ridge'].intercept_

    # 결과 출력
    y_label = f"log({label})→exp" if log_y else label
    print(f"\n{'='*55}")
    print(f"  {y_label}  |  최적 α = {alpha:.4f}")
    print(f"  In-sample R² = {insample_r2:.4f}")
    print(f"  LOOCV RMSE   = {loo_rmse:.2f}   LOOCV R² = {loo_r2:.4f}")
    print(f"\n  적합 계수 (표준화 공간):")
    print(f"    절편 = {intercept:.4f}")
    for n, c in zip(feat_names, coefs):
        print(f"    {n:20s} = {c:+.4f}")

    # 플롯
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # 계수 막대
    ax = axes[0]
    colors = ['steelblue' if c > 0 else 'tomato' for c in coefs]
    ax.barh(feat_names, coefs, color=colors, edgecolor='k')
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('계수 (표준화 공간)')
    ax.set_title(f'Ridge 계수 — {y_label}\n(α={alpha:.4f})')

    # Fitted vs Actual
    ax2 = axes[1]
    ax2.scatter(y_raw, y_fit, color='steelblue', edgecolors='k', s=70, label='In-sample')
    ax2.scatter(y_raw, loo_pred, color='orange', edgecolors='k', s=70, marker='s', label='LOOCV')
    lo, hi = min(y_raw.min(), min(y_fit.min(), loo_pred.min())), \
             max(y_raw.max(), max(y_fit.max(), loo_pred.max()))
    ax2.plot([lo, hi], [lo, hi], 'r--')
    ax2.set_xlabel('Actual')
    ax2.set_ylabel('Predicted')
    ax2.set_title(f'Fitted/LOOCV vs Actual\nIn-R²={insample_r2:.3f}  LOO-R²={loo_r2:.3f}')
    ax2.legend()
    plt.tight_layout()
    plt.show()

    return {'alpha': alpha, 'coefs': coefs, 'feat_names': feat_names,
            'insample_r2': insample_r2, 'loo_rmse': loo_rmse, 'loo_r2': loo_r2}

### 2-1. GD 방호성능

In [ ]:
y_GD = df['y_GD'].values

ridge_GD_raw = fit_ridge_poly(X_raw, y_GD, log_y=False, label='GD')
ridge_GD_log = fit_ridge_poly(X_raw, y_GD, log_y=True,  label='GD')

### 2-2. HD 방호성능

In [ ]:
y_HD = df['y_HD'].values

ridge_HD_raw = fit_ridge_poly(X_raw, y_HD, log_y=False, label='HD')
ridge_HD_log = fit_ridge_poly(X_raw, y_HD, log_y=True,  label='HD')

---
## 3. 가우시안 프로세스 회귀 (GPR)

- **ARD 커널**: 각 특성에 독립 length scale → 기여도 분리
- **전체 9개** 데이터로 회귀 적합 (in-sample)
- **유의성**: ΔlogLML = logLML(전체) − logLML(특성 제거)
  - ΔlogLML > 3 → 유의 / 1~3 → 경계 / < 1 → 비유의
- **SNR**: Signal variance / Noise level (높을수록 신호 명확)

In [ ]:
def fit_gpr(X_sc, y, n_feat=2, n_restarts=20):
    """ARD RBF + WhiteKernel GPR 적합."""
    kernel = (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=np.ones(n_feat),
              length_scale_bounds=[(1e-2, 1e3)] * n_feat)
        + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))
    )
    gpr = GaussianProcessRegressor(
        kernel=kernel, n_restarts_optimizer=n_restarts,
        normalize_y=True, random_state=42
    )
    gpr.fit(X_sc, y)
    return gpr


def fit_gpr_1d(X_sc_1d, y, n_restarts=20):
    """1D RBF + WhiteKernel (특성 1개 제거 시)."""
    kernel = (
        C(1.0, (1e-3, 1e3))
        * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))
        + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e3))
    )
    gpr = GaussianProcessRegressor(
        kernel=kernel, n_restarts_optimizer=n_restarts,
        normalize_y=True, random_state=42
    )
    gpr.fit(X_sc_1d, y)
    return gpr


def gpr_analysis(X_raw, y_raw, log_y, feat_names, y_label, n_restarts=20):
    """
    GPR 회귀 적합 + 유의성 검정 + 응답 표면.
    log_y=True 이면 log(y)를 학습하고 exp로 역변환해 표시.
    """
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    y      = np.log(y_raw) if log_y else y_raw

    # --- 전체 모형 적합 ---
    gpr = fit_gpr(X_sc, y, n_feat=X_raw.shape[1], n_restarts=n_restarts)
    lml_full = gpr.log_marginal_likelihood_value_

    # ARD length scales
    ls      = gpr.kernel_.k1.k2.length_scale
    signal  = gpr.kernel_.k1.k1.constant_value
    noise   = gpr.kernel_.k2.noise_level
    relev   = (1.0 / ls) / (1.0 / ls).sum() * 100

    # --- LML 유의성 검정 ---
    sig_rows = []
    for i, fname in enumerate(feat_names):
        idx = [j for j in range(len(feat_names)) if j != i]
        gpr_red  = fit_gpr_1d(X_sc[:, idx], y, n_restarts)
        lml_red  = gpr_red.log_marginal_likelihood_value_
        delta    = lml_full - lml_red
        sig      = '*** 유의' if delta >= 3 else ('*   경계' if delta >= 1 else '    비유의')
        sig_rows.append({'특성': fname,
                         'length scale (l)': round(ls[i], 4),
                         '기여도 (%)': round(relev[i], 2),
                         'ΔlogLML': round(delta, 4),
                         '유의성': sig})

    sig_df = pd.DataFrame(sig_rows)

    # --- 결과 출력 ---
    title_str = f"log({y_label})→exp" if log_y else y_label
    print(f"\n{'='*55}")
    print(f"  목표: {title_str}")
    print(f"  Signal variance σ² = {signal:.4f}")
    print(f"  Noise level   σ²_n = {noise:.4f}")
    print(f"  SNR = {signal/noise:.2f}")
    print(f"  logLML (전체) = {lml_full:.4f}")
    print()
    display(sig_df)

    # --- 적합값 플롯 + 응답 표면 ---
    y_fit, y_std = gpr.predict(X_sc, return_std=True)
    if log_y:
        y_plot_true = y_raw
        y_plot_fit  = np.exp(y_fit)
        y_lo = np.exp(y_fit - 1.96 * y_std)
        y_hi = np.exp(y_fit + 1.96 * y_std)
        y_err = [y_plot_fit - y_lo, y_hi - y_plot_fit]
    else:
        y_plot_true = y_raw
        y_plot_fit  = y_fit
        y_err = 1.96 * y_std

    insample_rmse = np.sqrt(mean_squared_error(y_plot_true, y_plot_fit))
    insample_r2   = r2_score(y_plot_true, y_plot_fit)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.errorbar(y_plot_true, y_plot_fit, yerr=y_err,
                fmt='o', color='steelblue', ecolor='lightblue',
                elinewidth=2, capsize=4, markersize=7)
    lo = min(y_plot_true.min(), y_plot_fit.min())
    hi = max(y_plot_true.max(), y_plot_fit.max())
    ax.plot([lo, hi], [lo, hi], 'r--')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Fitted')
    ax.set_title(f"GPR Fitted vs Actual\n{title_str}\nRMSE={insample_rmse:.2f}  R²={insample_r2:.4f}")

    ax2 = axes[1]
    ng = 60
    g1 = np.linspace(X_raw[:, 0].min() * 0.95, X_raw[:, 0].max() * 1.05, ng)
    g2 = np.linspace(X_raw[:, 1].min() * 0.95, X_raw[:, 1].max() * 1.05, ng)
    xx, yy = np.meshgrid(g1, g2)
    Xg = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    zz = gpr.predict(Xg).reshape(ng, ng)
    if log_y:
        zz = np.exp(zz)
    cp = ax2.contourf(xx, yy, zz, levels=20, cmap='RdYlGn')
    plt.colorbar(cp, ax=ax2, label=y_label)
    ax2.scatter(X_raw[:, 0], X_raw[:, 1], c='k', s=60, zorder=5)
    ax2.set_xlabel(feat_names[0])
    ax2.set_ylabel(feat_names[1])
    ax2.set_title(f'응답 표면 — {title_str}')

    plt.tight_layout()
    plt.show()

    return {'gpr': gpr, 'sig_df': sig_df,
            'insample_r2': insample_r2, 'insample_rmse': insample_rmse, 'lml': lml_full}

### 3-1. GD 방호성능

In [ ]:
gpr_GD_raw = gpr_analysis(X_raw, y_GD, log_y=False, feat_names=feat_names, y_label='GD')

In [ ]:
gpr_GD_log = gpr_analysis(X_raw, y_GD, log_y=True,  feat_names=feat_names, y_label='GD')

### 3-2. HD 방호성능

In [ ]:
gpr_HD_raw = gpr_analysis(X_raw, y_HD, log_y=False, feat_names=feat_names, y_label='HD')

In [ ]:
gpr_HD_log = gpr_analysis(X_raw, y_HD, log_y=True,  feat_names=feat_names, y_label='HD')

---
## 4. 결과 종합 비교

In [ ]:
# OLS 최적 모형 요약
print("=== OLS 최적 모형 ===")
ols_summary = pd.DataFrame([
    {'목표': 'GD', '모형': best_label_GD,
     'R²': round(best_res_GD.rsquared, 4),
     'adj-R²': round(best_res_GD.rsquared_adj, 4),
     'AIC': round(best_res_GD.aic, 2),
     'F p-값': f"{best_res_GD.f_pvalue:.4f}"},
    {'목표': 'HD', '모형': best_label_HD,
     'R²': round(best_res_HD.rsquared, 4),
     'adj-R²': round(best_res_HD.rsquared_adj, 4),
     'AIC': round(best_res_HD.aic, 2),
     'F p-값': f"{best_res_HD.f_pvalue:.4f}"},
])
display(ols_summary)

# Ridge 요약
print("\n=== Ridge (LOOCV 예측력) ===")
ridge_summary = pd.DataFrame([
    {'목표': 'GD (raw)', 'α': round(ridge_GD_raw['alpha'],4),
     'LOO RMSE': round(ridge_GD_raw['loo_rmse'],2), 'LOO R²': round(ridge_GD_raw['loo_r2'],4)},
    {'목표': 'GD (log)', 'α': round(ridge_GD_log['alpha'],4),
     'LOO RMSE': round(ridge_GD_log['loo_rmse'],2), 'LOO R²': round(ridge_GD_log['loo_r2'],4)},
    {'목표': 'HD (raw)', 'α': round(ridge_HD_raw['alpha'],4),
     'LOO RMSE': round(ridge_HD_raw['loo_rmse'],2), 'LOO R²': round(ridge_HD_raw['loo_r2'],4)},
    {'목표': 'HD (log)', 'α': round(ridge_HD_log['alpha'],4),
     'LOO RMSE': round(ridge_HD_log['loo_rmse'],2), 'LOO R²': round(ridge_HD_log['loo_r2'],4)},
])
display(ridge_summary)

# GPR 유의성 요약
print("\n=== GPR 유의성 (ΔlogLML) ===")
print("ΔlogLML > 3 → 유의  |  1~3 → 경계  |  < 1 → 비유의")
gpr_cases = [
    ('GD (raw)', gpr_GD_raw),
    ('GD (log)', gpr_GD_log),
    ('HD (raw)', gpr_HD_raw),
    ('HD (log)', gpr_HD_log),
]
gpr_rows = []
for label, res in gpr_cases:
    for _, row in res['sig_df'].iterrows():
        gpr_rows.append({'목표': label, '특성': row['특성'],
                         'ΔlogLML': row['ΔlogLML'],
                         '기여도(%)': row['기여도 (%)'],
                         '유의성': row['유의성']})
display(pd.DataFrame(gpr_rows))